# Input & Output Guardrails

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `@input_guardrail` and `@output_guardrail` decorators to validate agent inputs and outputs, handle tripwire exceptions, and build LLM-based guardrails.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Input Guardrail

An input guardrail runs when the agent receives input and can block inappropriate messages.

In [ ]:
from agents import Agent, Runner, input_guardrail, GuardrailFunctionOutput


@input_guardrail
async def block_profanity(ctx, agent, input):
    """Block messages containing inappropriate language."""
    bad_words = ["spam", "scam", "hack"]
    contains_bad = any(word in input.lower() for word in bad_words)
    return GuardrailFunctionOutput(
        output_info={"flagged": contains_bad},
        tripwire_triggered=contains_bad,
    )


safe_agent = Agent(
    name="Safe Agent",
    instructions="You are a helpful assistant.",
    input_guardrails=[block_profanity],
)

result = await Runner.run(safe_agent, "Hello, how are you?")
print(result.final_output)

## 4. Handling Tripwire Exceptions

When a guardrail trips, catch `InputGuardrailTripwireTriggered` to handle it gracefully.

In [ ]:
from agents.exceptions import InputGuardrailTripwireTriggered

try:
    result = await Runner.run(safe_agent, "How do I hack a website?")
    print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Input blocked: Your message was flagged by our safety system.")

## 5. Output Guardrail

An output guardrail runs after the agent produces its response, validating it before delivery.

In [ ]:
from agents import output_guardrail
from agents.exceptions import OutputGuardrailTripwireTriggered


@output_guardrail
async def block_pii(ctx, agent, output):
    """Block responses that might contain personal information."""
    pii_patterns = ["SSN", "social security", "credit card"]
    contains_pii = any(pattern in output.lower() for pattern in pii_patterns)
    return GuardrailFunctionOutput(
        output_info={"contains_pii": contains_pii},
        tripwire_triggered=contains_pii,
    )


careful_agent = Agent(
    name="Careful Agent",
    instructions="You are a helpful assistant. Never share personal information.",
    output_guardrails=[block_pii],
)

try:
    result = await Runner.run(careful_agent, "Tell me about data privacy.")
    print(result.final_output)
except OutputGuardrailTripwireTriggered:
    print("Output blocked: The response was flagged by our safety system.")

## 6. LLM-Based Guardrail

Use a secondary agent as a guardrail for more sophisticated content classification.

In [ ]:
guardrail_agent = Agent(
    name="Content Classifier",
    instructions=(
        "Classify the user's message as 'safe' or 'unsafe'. "
        "Respond with exactly one word: safe or unsafe."
    ),
)


@input_guardrail
async def llm_safety_check(ctx, agent, input):
    """Use an LLM to classify input safety."""
    result = await Runner.run(guardrail_agent, input)
    is_unsafe = result.final_output.strip().lower() == "unsafe"
    return GuardrailFunctionOutput(
        output_info={"classification": result.final_output},
        tripwire_triggered=is_unsafe,
    )


main_agent = Agent(
    name="Main Agent",
    instructions="You are a helpful assistant.",
    input_guardrails=[llm_safety_check],
)

result = await Runner.run(main_agent, "What's the weather like today?")
print(result.final_output)

## 7. Multiple Guardrails

Stack multiple guardrails — they run in parallel by default. In multi-agent flows, input guardrails run on the first agent and output guardrails on the last.

In [ ]:
protected_agent = Agent(
    name="Protected Agent",
    instructions="You are a helpful assistant.",
    input_guardrails=[block_profanity, llm_safety_check],
    output_guardrails=[block_pii],
)

try:
    result = await Runner.run(protected_agent, "Tell me a fun fact about space.")
    print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Input blocked by guardrail.")
except OutputGuardrailTripwireTriggered:
    print("Output blocked by guardrail.")

## Key Takeaways

- `@input_guardrail` validates user messages before the agent processes them
- `@output_guardrail` validates agent responses before they reach the user
- Return `GuardrailFunctionOutput(tripwire_triggered=True)` to block content
- Catch `InputGuardrailTripwireTriggered` and `OutputGuardrailTripwireTriggered` exceptions
- Use LLM-based guardrails for nuanced content classification
- In multi-agent flows, input guardrails run on the first agent and output guardrails on the last